# Agentic Enterprise Architecture

## Scenario: operate an ecosystem of agents, tools, and MCP services

Northstar wants to register a new high-risk customer-impact agent and a metrics MCP server. This notebook teaches the platform controls that make reuse possible without shadow discovery, credential inheritance, policy bypass, unbounded cost, or missing audit evidence.

**Outcomes:** build an agent/tool registry, use identity-bound discovery, distinguish catalog from authorization, add observability/evaluation/FinOps, and define lifecycle gates.

![Agentic enterprise architecture](../../../assets/agentic-enterprise-architecture.svg)

The control plane owns policy, identity, registration, budgets, audits, and promotion. Individual agents cannot self-authorize a tool or publish themselves into the trusted catalog.

## 1. Register accountable capabilities

An enterprise catalog answers: *what is this capability, who owns it, what can it do, what data and tools may it access, which version is running, has it passed release evaluation, and how can it be revoked?* Registration is a policy gate, not a convenience list.

In [1]:
from lab import ControlPlane, AgentRecord, ToolRecord

platform = ControlPlane()
platform.register_agent(AgentRecord('customer-impact', 'commerce-platform', ('impact-analysis',), 'high', '1.2.0', True))
platform.register_tool(ToolRecord('metrics-mcp', 'mcp', 'data-platform', ('impact-analysis',), True, '4.1.0'))
print(platform.agents)
print(platform.tools)

{'customer-impact': AgentRecord(name='customer-impact', owner='commerce-platform', capabilities=('impact-analysis',), risk='high', version='1.2.0', eval_passed=True)}
{'metrics-mcp': ToolRecord(name='metrics-mcp', kind='mcp', owner='data-platform', scopes=('impact-analysis',), trusted=True, version='4.1.0')}


## 2. Discover is not authorization

Discovery finds an approved compatible capability. Authorization still evaluates the requesting workload identity, tenant, purpose, exact scope, risk, time, budget, and approval state. An MCP or A2A descriptor helps interoperable discovery but does not grant trust by itself. Treat tool schemas as versioned interfaces with owners, provenance, evaluation, and revocation paths.

In [2]:
tools = platform.discover('customer-impact', 'impact-analysis', 'tenant:acme')
print('approved discovery:', tools)
proposal = platform.execute('customer-impact', 'metrics-mcp', 'impact-analysis', cost_cents=8)
print(proposal)
assert proposal == 'proposal-only: human approval required'
print('\n'.join(platform.audit))

approved discovery: ['metrics-mcp']
proposal-only: human approval required
agent-registered:customer-impact:1.2.0
tool-registered:metrics-mcp:4.1.0
discover:customer-impact:impact-analysis:tenant:acme:metrics-mcp


## 3. Knowledge, orchestration, and operations

Context and memory are also enterprise assets. Apply tenant and purpose filters, source provenance, data classification, freshness, retention, and deletion before relevance ranking. The orchestrator selects the least autonomous path and establishes trace IDs, tool/time/cost/delegation budgets, retries, approval interrupts, and safe terminal states.

Observability joins agent version, policy, identity, tool scope, outcome, and approval without storing unnecessary sensitive payloads. Evaluation gates new versions and regressions. FinOps attributes cost to the agent, tenant, business unit, and accepted outcome; it constrains spend and makes cost-per-success visible.

## 4. Failure drills and exercises

1. Attempt to register a tool with `trusted=False`; explain why the registry fails closed.
2. Add a version-promotion method that requires fresh evaluation evidence.
3. Add tenant/purpose to `execute` and reject a cross-tenant request.
4. Model a 25-cent task budget and compare cost per accepted outcome for two agents.
5. Design emergency revocation: registry disable, token invalidation, orchestrator fallback, audit event, and incident owner.

## References

- [MCP enterprise-managed authorization](https://modelcontextprotocol.io/extensions/auth/enterprise-managed-authorization)
- [A2A agent discovery](https://a2a-protocol.org/latest/topics/agent-discovery/)
- [Survey of AI agent registry solutions](https://arxiv.org/abs/2508.03095)
- [OpenAI Frontier](https://openai.com/business/frontier/)
- [SAGA governance architecture](https://arxiv.org/abs/2504.21034)